# MV-VTON — пилот на пальто Silver.Vy

Ноутбук готовит изображения для тестового загрузчика MV-VTON и запускает одну генерацию. Используются один фронтальный снимок человека, полный фронт пальто, крупный фронтальный ракурс и боковой ракурс. Код модели использует фронтальный и второй ракурс как замену заднего; боковой ракурс не равен заднему, поэтому такой запуск — эксперимент, не проверка штатного режима.

В Colab выбери **Среда выполнения → Сменить среду выполнения → T4 GPU**. Три фотографии пальто ноутбук скачает из публичного репозитория Silver.Vy. Загрузи только `person.jpg`; для первого теста подойдёт подготовленный синтетический человек.

In [ ]:
!if [ ! -d /content/MV-VTON/.git ]; then git clone -q https://github.com/hywang2002/MV-VTON.git /content/MV-VTON; fi
%cd /content/MV-VTON
!pip -q install numpy==1.26.4 pytorch-lightning==1.9.5 omegaconf==2.1.1 einops==0.4.1 opencv-python-headless==4.8.1.78 transformers==4.39.3 controlnet-aux gdown
!pip -q install git+https://github.com/openai/CLIP.git

После установки перезапусти среду Colab, чтобы перезагрузились NumPy и бинарные модули. Затем продолжай с проверки GPU. Если окружение сбросило файлы, выполни установку ещё раз и снова перезапусти среду.

In [ ]:
import torch
import numpy as np
if np.__version__ != '1.26.4':
    raise RuntimeError(f'Ожидался NumPy 1.26.4, установлен {np.__version__}. Перезапусти среду после установки.')
if not torch.cuda.is_available():
    raise RuntimeError('GPU не включен. Выбери T4 GPU в настройках среды выполнения.')
print(torch.cuda.get_device_name(0), torch.__version__, np.__version__)

In [ ]:
from google.colab import files
import urllib.request, urllib.parse
uploaded = files.upload()
if 'person.jpg' not in uploaded:
    raise ValueError('Загрузи фотографию человека с именем person.jpg')
base = 'https://raw.githubusercontent.com/SilverVy/silvervy-shop/main/'
coat_files = {
    'coat-front.png': 'server/public/Одежда/coat/2_coat.png',
    'coat-detail.png': 'server/public/Одежда/coat/1_coat.png',
    'coat-side.png': 'server/public/Одежда/coat/3_coat.png',
}
for name, path in coat_files.items():
    url = base + urllib.parse.quote(path, safe='/')
    urllib.request.urlretrieve(url, '/content/' + name)
print('Загружено пальто:', list(coat_files))

In [ ]:
import os, shutil, json
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
import numpy as np
if not hasattr(np, 'stringType'):
    np.stringType = np.bytes_
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from controlnet_aux import OpenposeDetector
import torch.nn.functional as F

ROOT = Path('/content/MV-VTON')
DATA = Path('/content/MVG/test')
PERSON_ID = '00001_1'
CLOTH_ID = '00001_1'
model_id = 'mattmdjaga/segformer_b2_clothes'
processor = SegformerImageProcessor.from_pretrained(model_id)
parser = SegformerForSemanticSegmentation.from_pretrained(model_id).to('cuda').eval()
pose_detector = OpenposeDetector.from_pretrained('lllyasviel/Annotators')

def open_rgb(path):
    image = ImageOps.exif_transpose(Image.open(path))
    if image.mode == 'P' and 'transparency' in image.info:
        image = image.convert('RGBA')
    if 'A' in image.getbands():
        background = Image.new('RGBA', image.size, (128, 128, 128, 255))
        image = Image.alpha_composite(background, image)
    return image.convert('RGB')

def parse_mask(image, class_ids):
    inputs = processor(images=image, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        logits = parser(**inputs).logits
        logits = F.interpolate(logits, size=(image.height, image.width), mode='bilinear', align_corners=False)[0]
        probabilities = logits.softmax(dim=0)
        confidence, labels = probabilities.max(dim=0)
    selected = torch.zeros_like(labels, dtype=torch.bool)
    for class_id in class_ids:
        selected |= labels == class_id
    return Image.fromarray(((selected & (confidence >= 0.42)).byte().cpu().numpy() * 255).astype('uint8'))

def frame(image, fill):
    target_width, target_height = 384, 512
    target_ratio = target_width / target_height
    if image.width / image.height > target_ratio:
        crop_width = round(image.height * target_ratio)
        left = (image.width - crop_width) // 2
        image = image.crop((left, 0, left + crop_width, image.height))
    else:
        canvas_width = round(image.height * target_ratio)
        canvas = Image.new(image.mode, (canvas_width, image.height), fill)
        canvas.paste(image, ((canvas_width-image.width)//2, 0))
        image = canvas
    return image.resize((target_width, target_height), Image.Resampling.LANCZOS)

for folder in ['image-wo-bg', 'cloth', 'cloth-mask', 'inpaint_mask', 'skeletons', 'warp_feat_unpair']:
    (DATA / folder).mkdir(parents=True, exist_ok=True)
person_dir = DATA / 'image-wo-bg' / PERSON_ID
cloth_dir = DATA / 'cloth' / CLOTH_ID
cloth_mask_dir = DATA / 'cloth-mask' / CLOTH_ID
for folder in [person_dir, cloth_dir, cloth_mask_dir, DATA / 'inpaint_mask' / PERSON_ID, DATA / 'skeletons' / PERSON_ID, DATA / 'skeletons' / CLOTH_ID]:
    folder.mkdir(parents=True, exist_ok=True)

person = open_rgb('/content/person.jpg')
person_mask = parse_mask(person, {4, 6, 14, 15})
person_mask_fraction = np.asarray(person_mask).mean() / 255
if not 0.01 <= person_mask_fraction <= 0.45:
    raise ValueError(f'Подозрительная маска человека: {person_mask_fraction:.3f}. Проверь сегментацию.')
person_512 = frame(person, (128, 128, 128))
person_mask_512 = frame(person_mask, 0)
agnostic = person_512.copy()
agnostic.paste((128, 128, 128), mask=person_mask_512)
agnostic.save(DATA / 'warp_feat_unpair' / f'{PERSON_ID}.jpg')
person_512.save(person_dir / '2.png')
person_512.save(person_dir / '1.png')
person_512.save(person_dir / '5.png')
person_mask_512.save(DATA / 'inpaint_mask' / PERSON_ID / '2.png')
pose_person = pose_detector(person_512)
pose_person.save(DATA / 'skeletons' / PERSON_ID / '2.png')
pose_person.save(DATA / 'skeletons' / PERSON_ID / '1.png')
pose_person.save(DATA / 'skeletons' / PERSON_ID / '5.png')

views = [('coat-front.png', '1.png'), ('coat-detail.png', '2.png'), ('coat-side.png', '5.png')]
for filename, output_name in views:
    garment = open_rgb('/content/' + filename)
    garment_mask = parse_mask(garment, {4, 7})
    garment_mask_fraction = np.asarray(garment_mask).mean() / 255
    if not 0.02 <= garment_mask_fraction <= 0.85:
        raise ValueError(f'Подозрительная маска {filename}: {garment_mask_fraction:.3f}. Проверь сегментацию.')
    garment_512 = frame(garment, (128, 128, 128))
    garment_mask_512 = frame(garment_mask, 0)
    garment_for_model = garment_512.copy()
    garment_for_model.paste((128, 128, 128), mask=Image.eval(garment_mask_512, lambda value: 255-value))
    garment_for_model.save(cloth_dir / output_name)
    garment_mask_512.save(cloth_mask_dir / output_name)
    pose = pose_detector(garment_512)
    pose.save(DATA / 'skeletons' / CLOTH_ID / output_name)
    preview = garment_512.convert('RGBA')
    tint = Image.new('RGBA', preview.size, (30, 210, 140, 0))
    tint.putalpha(garment_mask_512.point(lambda value: value * 110 // 255))
    Image.alpha_composite(preview, tint).convert('RGB').save('/content/' + output_name.replace('.png', '-mask-review.jpg'))

(DATA.parent / 'unpaired.txt').write_text(f'{PERSON_ID.split("_")[0]} {CLOTH_ID.split("_")[0]}\n')
print('Подготовка завершена. Перед генерацией проверь mask-review.jpg и person-agnostic.')

## Проверка масок

Перед запуском посмотри все три `*-mask-review.jpg` и маску человека. Для этого пилота маска человека включает верхнюю одежду, брюки и руки, чтобы длинное пальто могло закрыть торс и ноги; лицо, кисти и обувь не должны попасть внутрь. Сегментатор может спутать длинное пальто с платьем или захватить одежду под ним. Если маски явно неверны, генерацию не запускай. Автоматическая маска — черновик, не подтверждение правильной сегментации.

In [ ]:
from IPython.display import display
for name in ['1-mask-review.jpg', '2-mask-review.jpg', '5-mask-review.jpg']:
    display(Image.open('/content/' + name))
display(Image.open(DATA / 'warp_feat_unpair' / f'{PERSON_ID}.jpg'))

In [ ]:
!cp ldm/data/cp_dataset_mv_unpaired.py ldm/data/cp_dataset.py
from pathlib import Path
dataset_code = Path('ldm/data/cp_dataset.py')
source = dataset_code.read_text()
old_width = 'self.fine_width = int(image_size / 256 * 256)'
if old_width not in source:
    raise RuntimeError('Ожидаемая строка размера в MV-VTON не найдена; не запускаю с несовпадающим размером.')
dataset_code.write_text(source.replace(old_width, 'self.fine_width = int(image_size / 256 * 192)'))
!mkdir -p models/vgg checkpoints
!gdown -q 1rvow8jStPt8t2prDcSRlnf8yzXhrYeGo -O models/vgg/vgg19_conv.pth
!gdown -q 1J91PoT8A9yqHWNxkgRe6ZCnDEhN-H9O6 -O checkpoints/mvg.ckpt
!python test.py --gpu_id 0 --ddim_steps 50 --outdir /content/mv-vton-results --config configs/viton512.yaml --dataroot /content/MVG --ckpt checkpoints/mvg.ckpt --n_samples 1 --seed 23 --scale 1 --H 512 --W 384

In [ ]:
from glob import glob
from IPython.display import display
result_files = sorted(glob('/content/mv-vton-results/**/*.*', recursive=True))
result_images = [path for path in result_files if path.lower().endswith(('.png', '.jpg', '.jpeg'))]
if not result_images:
    raise FileNotFoundError('MV-VTON не создал изображение. Проверь полный traceback запуска.')
for path in result_images[-3:]:
    print(path)
    display(Image.open(path))

## Как оценивать результат

Сначала оцени только форму и сохранение пальто. MV-VTON не принимает размер пользователя и не обещает точную посадку. Не сравнивай размеры по тексту: для размерного слоя сначала потребуются реальные мерки пальто по каждому размеру и мерки тела, затем отдельная геометрическая коррекция. Если установка падает из-за совместимости старого репозитория с текущим Colab, сохрани полный traceback — не переустанавливай случайные версии CUDA и PyTorch поверх работающей среды.